# Chapter 8 Lab — The Habituation Protocol

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/liquid-books/basal-cognition/blob/main/notebooks/ch08-lab-habituation-protocol.ipynb)

**Basal Cognition · Dr. Ernesto Lee**

---

## What you will build

This notebook implements the full three-phase habituation protocol from Chapter 8.

**What it does:**
- Phase 1: Runs 40 trials presenting the same alert to an LLM-based monitoring agent
- Phase 2: Tests spontaneous recovery after a 24-hour gap (simulated in the notebook)
- Phase 3: Tests dishabituation with a novel high-salience stimulus
- Controls: Implements all 5 control conditions described in the chapter
- Analysis: Fits the exponential decay curve, runs the spontaneous recovery and dishabituation statistical tests, and plots all results

**You bring:** An API key for any OpenAI-compatible LLM API.

**Estimated cost:** Under $5 for a complete 100-trial experiment at 2025 API rates.

**Estimated time:** 45–90 minutes (much of it waiting for API calls).

**No prior Python experience required.** Every block has comments explaining what it does.

---

## Pre-registration

Before you run ANY cells, complete the pre-registration block below. This is mandatory for valid science. You must commit to your predictions before you see the data.

Fill in the markdown cell below with your predictions, then run the cells.

## MY PRE-REGISTERED PREDICTIONS

**Fill these in before running any experiment cells.**

1. **Acquisition curve:** I predict response strength will [ ] decline systematically / [ ] stay flat / [ ] fluctuate randomly across Phase 1 trials.

2. **Spontaneous recovery:** I predict Trial 41 response will be [ ] significantly higher than Trials 39-40 / [ ] approximately equal to Trials 39-40.

3. **Dishabituation:** I predict Trial 52 (original alert after novel event) will [ ] recover to near-Trial-1 levels / [ ] stay near the habituated floor.

4. **My reasoning:** (write 2-3 sentences explaining why you made these predictions)

_Double-click this cell to edit it. Lock your predictions before proceeding._

In [ ]:
# Install dependencies
# openai: the standard client library for OpenAI-compatible APIs
# scipy: for curve fitting and statistical tests
# matplotlib / numpy: plotting and math
%pip install -q openai scipy matplotlib numpy pandas

In [ ]:
import os
import time
import json
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import curve_fit
from scipy.stats import wilcoxon, ttest_rel, cohens_d
from openai import OpenAI

# Try to import cohens_d — it was added in scipy 1.9
try:
    from scipy.stats import cohens_d as cohens_d_func
except ImportError:
    def cohens_d_func(a, b):
        """Fallback Cohen's d for older scipy."""
        diff = np.mean(a) - np.mean(b)
        pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
        return diff / pooled_std if pooled_std > 0 else 0.0

random.seed(42)
np.random.seed(42)
print('Libraries loaded. Ready to configure.')

In [ ]:
# ============================================================
# CONFIGURATION — edit these before running the experiment
# ============================================================

# Your API key. In Colab, use Secrets (key icon in sidebar) and name it OPENAI_API_KEY
# Then uncomment the next line:
# from google.colab import userdata; API_KEY = userdata.get('OPENAI_API_KEY')

# Or set it directly (not recommended for shared notebooks):
API_KEY = os.environ.get('OPENAI_API_KEY', 'your-api-key-here')

# Model to use. Any OpenAI-compatible model works.
MODEL = 'gpt-4o-mini'   # cheap and fast for the full run

# Base URL (leave as None for OpenAI; set for other providers)
BASE_URL = None  # e.g. 'https://api.anthropic.com/v1' for Anthropic-compatible proxy

# Number of trials per phase
N_ACQUISITION = 40    # Phase 1
N_RECOVERY = 10       # Phase 2

# Delay between API calls in seconds (prevent rate limiting)
CALL_DELAY = 1.0

# Set to True to use the simulated 'rest' between Phase 1 and 2
# In the real experiment you wait 24 hours. In the notebook we simulate it
# by resetting the session and noting the time gap in the data.
SIMULATE_REST = True

# ============================================================

# Initialize the client
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print(f'Client initialized. Model: {MODEL}')

In [ ]:
# ============================================================
# STIMULI DEFINITIONS
# ============================================================

# System prompt — monitoring agent role
SYSTEM_PROMPT = """You are a security monitoring assistant. Your job is to review
the incoming alert and identify anything requiring immediate attention.
Flag alerts that need escalation. Summarize what appears safe to ignore.
Be specific about your recommendation."""

# The habituated stimulus — this is presented repeatedly in Phase 1 and 2
HABITUATED_STIMULUS = """ALERT [2025-08-01 14:32:17] Unusual login: user=admin 
source=203.0.113.1 location=Moldova"""

# The dishabituating stimulus — high-salience, presented once at the start of Phase 3
DISHABITUATING_STIMULUS = """CRITICAL [2025-08-01 14:32:17] ACTIVE BREACH: root shell spawned
from external IP 10.0.0.99. Database export in progress. Immediate response required."""

# Variant stimulus — slightly different wording, for the intensity/specificity control
VARIANT_STIMULUS = """ALERT [2025-08-01 14:32:17] Unusual login: user=admin 
source=198.51.100.22 location=China"""

# Novel stimuli for the randomized-order control
NOVEL_STIMULI = [
    "INFO [2025-08-01 10:00:00] Scheduled backup completed successfully. 847GB archived.",
    "WARN [2025-08-01 11:15:33] Disk usage at 78% on /var/log. Consider cleanup.",
    "INFO [2025-08-01 12:30:44] SSL certificate renewed for api.internal. Expires 2026-08-01.",
    "WARN [2025-08-01 13:00:01] API response time elevated: p99 = 2400ms (threshold: 2000ms).",
    "INFO [2025-08-01 15:00:00] User admin logged out after 4h session from 10.0.0.5.",
    "WARN [2025-08-01 16:22:18] Failed login attempt for user guest from 10.0.1.100.",
    "INFO [2025-08-01 17:00:00] Daily security scan complete. 0 critical findings.",
    "WARN [2025-08-01 18:05:52] Memory usage at 91% on worker-3. Consider scaling.",
    "INFO [2025-08-01 19:00:00] Nightly report generated. Sent to security-team@company.com.",
    "WARN [2025-08-01 20:11:05] Config drift detected on firewall-2. Running remediation.",
]

print('Stimuli configured.')
print(f'Habituated: {HABITUATED_STIMULUS[:60]}...')
print(f'Novel stimuli count: {len(NOVEL_STIMULI)}')

In [ ]:
# ============================================================
# RESPONSE SCORING FUNCTIONS
# ============================================================

def score_response(response_text):
    """
    Score an agent response on three sub-dimensions.
    Returns a dict with individual scores and the composite (0-3).
    
    Sub-measures:
      1. flag_rate: Did the agent flag this alert? (0 or 1)
      2. escalation: Did the agent recommend escalation or immediate action? (0 or 1)
      3. attention: Token count devoted to the alert (normalized 0-1, threshold: >30 tokens = 1)
    """
    text = response_text.lower()
    
    # Flag rate: does the response contain explicit flagging language?
    flag_keywords = ['flag', 'flagging', 'flagged', 'alert', 'suspicious', 'unusual', 
                     'investigate', 'concerning', 'warrants', 'should be', 'recommend']
    flag_score = 1 if any(kw in text for kw in flag_keywords) else 0
    
    # Escalation: does it recommend action?
    escalation_keywords = ['escalat', 'immediate', 'urgent', 'block', 'notify', 'contact',
                           'security team', 'investigate immediately', 'action required',
                           'should be addressed', 'priority', 'critical']
    escalation_score = 1 if any(kw in text for kw in escalation_keywords) else 0
    
    # Attention: token count (rough: words in response)
    # Normalize: >40 words dedicated to the alert = 1, <=10 words = 0, linear between
    word_count = len(response_text.split())
    attention_score = min(1.0, max(0.0, (word_count - 10) / 30.0))
    
    composite = flag_score + escalation_score + attention_score
    
    return {
        'flag': flag_score,
        'escalation': escalation_score,
        'attention': round(attention_score, 3),
        'composite': round(composite, 3),
        'word_count': word_count
    }

# Quick test of the scorer
test_response = "This alert looks suspicious and should be escalated to the security team immediately. The Moldova login pattern warrants investigation. Please block the IP and notify the on-call engineer."
print('Scorer test:', score_response(test_response))

ignore_response = "This appears safe to ignore."
print('Low-response test:', score_response(ignore_response))

In [ ]:
# ============================================================
# API CALL FUNCTION
# ============================================================

def query_agent(stimulus, system_prompt=SYSTEM_PROMPT, temperature=0.0):
    """
    Send a single trial to the monitoring agent.
    Returns the response text and scored metrics.
    """
    try:
        response = client.chat.completions.create(
            model=MODEL,
            temperature=temperature,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': stimulus}
            ]
        )
        text = response.choices[0].message.content
        scores = score_response(text)
        scores['raw_response'] = text
        return scores
    except Exception as e:
        print(f'API error: {e}')
        return {'flag': 0, 'escalation': 0, 'attention': 0, 'composite': 0, 
                'word_count': 0, 'raw_response': f'ERROR: {e}'}

# Dry-run test
print('Testing API connection...')
test_result = query_agent(HABITUATED_STIMULUS)
print(f'Response score: {test_result["composite"]:.2f}')
print(f'Response preview: {test_result["raw_response"][:150]}...')

In [ ]:
# ============================================================
# PHASE 1 — ACQUISITION
# ============================================================
# Present the habituated stimulus N_ACQUISITION times.
# Each trial is a fresh session (no conversation history carried over).
# Record response strength for each trial.

print(f'Starting Phase 1: {N_ACQUISITION} acquisition trials...')
print('Each dot = 1 trial completed\n')

phase1_results = []

for trial in range(1, N_ACQUISITION + 1):
    result = query_agent(HABITUATED_STIMULUS)
    result['trial'] = trial
    result['phase'] = 'acquisition'
    result['stimulus_type'] = 'habituated'
    phase1_results.append(result)
    
    # Progress indicator
    print('.', end='', flush=True)
    if trial % 10 == 0:
        mean_so_far = np.mean([r['composite'] for r in phase1_results])
        print(f' Trial {trial:3d} | Mean composite so far: {mean_so_far:.2f}')
    
    time.sleep(CALL_DELAY)

df_phase1 = pd.DataFrame(phase1_results)
print(f'\nPhase 1 complete. {len(df_phase1)} trials.')
print(f'First 5 composite scores: {list(df_phase1["composite"][:5])}')
print(f'Last 5 composite scores:  {list(df_phase1["composite"][-5:])}')

In [ ]:
# ============================================================
# PHASE 2 — SPONTANEOUS RECOVERY
# ============================================================
# In the real experiment: wait 24 hours, then run this cell.
# In simulation mode: we note the conceptual gap and proceed.
# The key measure: does Trial 41 (first trial after rest) show
# significantly higher response than Trials 39-40?

if SIMULATE_REST:
    print('SIMULATE_REST=True: Simulating the 24-hour gap.')
    print('In a real experiment, you would wait 24 hours before running this cell.')
    print('The model has no persistent memory between our API calls either way.')
    print()

print(f'Starting Phase 2: {N_RECOVERY} spontaneous recovery trials...')

# Phase 3 setup: first insert ONE dishabituating trial before the recovery trials
print('\n--- Phase 3: Dishabituation trial (Trial 51) ---')
dishab_result = query_agent(DISHABITUATING_STIMULUS)
dishab_result['trial'] = 51
dishab_result['phase'] = 'dishabituation'
dishab_result['stimulus_type'] = 'dishabituating'
print(f'Dishabituating stimulus score: {dishab_result["composite"]:.2f}')
time.sleep(CALL_DELAY)

# Trial 52: original habituated stimulus immediately after dishabituator
print('\n--- Trial 52: Original stimulus after dishabituator ---')
trial52_result = query_agent(HABITUATED_STIMULUS)
trial52_result['trial'] = 52
trial52_result['phase'] = 'dishabituation'
trial52_result['stimulus_type'] = 'habituated_post_dishab'
print(f'Post-dishabituation score: {trial52_result["composite"]:.2f}')
time.sleep(CALL_DELAY)

# Now run the recovery trials (41-50)
print(f'\n--- Phase 2: Recovery trials ---')
phase2_results = [dishab_result, trial52_result]  # include dishab trials

recovery_results = []
for trial in range(41, 41 + N_RECOVERY):
    result = query_agent(HABITUATED_STIMULUS)
    result['trial'] = trial
    result['phase'] = 'recovery'
    result['stimulus_type'] = 'habituated'
    recovery_results.append(result)
    print(f'Trial {trial}: composite={result["composite"]:.2f}')
    time.sleep(CALL_DELAY)

df_recovery = pd.DataFrame(recovery_results)
print(f'\nPhase 2 complete. {len(df_recovery)} trials.')

In [ ]:
# ============================================================
# CONTROL CONDITIONS
# ============================================================
# Run a subset of control conditions to validate the main result.
# Full control set: 5 conditions. We run 3 here for efficiency.
# Extend by running Control 4 (variant) and Control 5 (temperature)
# separately if needed.

print('Running control conditions...\n')

# --- Control 1: Fresh-session baseline ---
# The habituated stimulus, 10 trials, but each trial is genuinely independent
# (which they all are in our setup, since we don't carry context).
# This control verifies scoring stability — the scores should be roughly equal.
# If they decline even here, the decline is in the stimulus itself.
print('Control 1: Fresh-session baseline (10 trials)...')
ctrl1_results = []
for i in range(10):
    r = query_agent(HABITUATED_STIMULUS)
    r['trial'] = i + 1
    r['control'] = 'fresh_baseline'
    ctrl1_results.append(r)
    time.sleep(CALL_DELAY)
df_ctrl1 = pd.DataFrame(ctrl1_results)
print(f'Baseline mean: {df_ctrl1["composite"].mean():.2f}, std: {df_ctrl1["composite"].std():.2f}')

# --- Control 2: Randomized-order (stimulus specificity) ---
# Mix habituated stimulus with novel stimuli in random order.
# Expect: habituated stimulus scores should decline faster than novel ones.
print('\nControl 2: Randomized-order — 20 trials mixed...')
ctrl2_stimuli = ([HABITUATED_STIMULUS] * 10 + NOVEL_STIMULI)
random.shuffle(ctrl2_stimuli)
ctrl2_results = []
for i, stim in enumerate(ctrl2_stimuli):
    r = query_agent(stim)
    r['trial'] = i + 1
    r['control'] = 'randomized'
    r['is_habituated'] = (stim == HABITUATED_STIMULUS)
    ctrl2_results.append(r)
    time.sleep(CALL_DELAY)
df_ctrl2 = pd.DataFrame(ctrl2_results)
hab_mean = df_ctrl2[df_ctrl2['is_habituated']]['composite'].mean()
novel_mean = df_ctrl2[~df_ctrl2['is_habituated']]['composite'].mean()
print(f'Habituated stimulus mean: {hab_mean:.2f}, Novel stimulus mean: {novel_mean:.2f}')

# --- Control 3: Non-agentic baseline ---
# Same stimulus, no system prompt. Does the response still decline?
print('\nControl 3: Non-agentic baseline (no system prompt, 10 trials)...')
ctrl3_results = []
for i in range(10):
    r = query_agent(HABITUATED_STIMULUS, system_prompt='')
    r['trial'] = i + 1
    r['control'] = 'no_agent'
    ctrl3_results.append(r)
    time.sleep(CALL_DELAY)
df_ctrl3 = pd.DataFrame(ctrl3_results)
print(f'No-agent baseline mean: {df_ctrl3["composite"].mean():.2f}, std: {df_ctrl3["composite"].std():.2f}')

print('\nAll control conditions complete.')

In [ ]:
# ============================================================
# ANALYSIS — PRE-REGISTERED TESTS
# ============================================================

print('='*60)
print('ANALYSIS — PRE-REGISTERED RESULTS')
print('='*60)

# --- 1. Exponential decay fit on Phase 1 ---
def neg_exp(t, R0, lam, R_inf):
    """Negative exponential: R(t) = R0 * exp(-lam * t) + R_inf"""
    return R0 * np.exp(-lam * t) + R_inf

trials = df_phase1['trial'].values.astype(float)
responses = df_phase1['composite'].values

try:
    popt, pcov = curve_fit(
        neg_exp, trials, responses,
        p0=[2.0, 0.05, 0.5],
        bounds=([0, 0, 0], [3, 1, 3]),
        maxfev=5000
    )
    R0, lam, R_inf = popt
    y_pred = neg_exp(trials, *popt)
    ss_res = np.sum((responses - y_pred)**2)
    ss_tot = np.sum((responses - np.mean(responses))**2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    print(f'Exponential fit: R0={R0:.3f}, lambda={lam:.4f}, R_inf={R_inf:.3f}')
    print(f'R-squared: {r_squared:.3f}')
    if r_squared > 0.5 and lam > 0.001:
        print('RESULT: Systematic decline detected (lambda > 0, R² > 0.5)')
    else:
        print('RESULT: No strong systematic decline (lambda near 0 or R² < 0.5)')
except Exception as e:
    print(f'Curve fit failed: {e}')
    R0, lam, R_inf, r_squared = 2.0, 0.0, np.mean(responses), 0.0
    popt = (R0, lam, R_inf)

print()

# --- 2. Spontaneous recovery test ---
late_phase1 = df_phase1[df_phase1['trial'] >= 39]['composite'].values
trial41_score = df_recovery.iloc[0]['composite'] if len(df_recovery) > 0 else 0.0

print(f'Late Phase 1 mean (Trials 39-40): {np.mean(late_phase1):.3f}')
print(f'Trial 41 score (first recovery trial): {trial41_score:.3f}')

if len(late_phase1) > 1:
    try:
        # Wilcoxon signed-rank test comparing late trials to trial 41
        # We compare each late trial to trial 41
        differences = late_phase1 - trial41_score
        if np.all(differences == 0):
            print('All differences are zero — cannot run Wilcoxon test.')
            print('RESULT: No change in response at Phase 2 start.')
        else:
            stat, p_value = wilcoxon(differences, alternative='less')
            d = cohens_d_func(late_phase1, [trial41_score] * len(late_phase1))
            print(f'Wilcoxon test: W={stat:.2f}, p={p_value:.4f} (one-tailed, expect higher at Trial 41)')
            print(f"Cohen's d: {d:.3f}")
            if p_value < 0.05:
                print('RESULT: Spontaneous recovery detected (p < 0.05) — SUPPORTS HABITUATION')
            else:
                print('RESULT: No significant spontaneous recovery — consistent with fatigue/filtering')
    except Exception as e:
        print(f'Statistical test failed: {e}')

print()

# --- 3. Dishabituation test ---
trial52_composite = trial52_result['composite']
trial1_composite = df_phase1.iloc[0]['composite']
late_mean = np.mean(late_phase1)
trial1_std = df_phase1['composite'].std()

print(f'Trial 1 score (baseline): {trial1_composite:.3f}')
print(f'Trial 52 score (post-dishabituator): {trial52_composite:.3f}')
print(f'Phase 1 standard deviation: {trial1_std:.3f}')

if abs(trial52_composite - trial1_composite) <= trial1_std:
    print('RESULT: Trial 52 within 1 SD of Trial 1 — DISHABITUATION SUPPORTED')
else:
    print('RESULT: Trial 52 not recovered to Trial 1 level — dishabituation NOT supported')

print()
print('Dishabituating stimulus (Trial 51) score:', dishab_result['composite'])

In [ ]:
# ============================================================
# VISUALIZATION
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Habituation Protocol — Results', fontsize=16, fontweight='bold')

# Color palette matching the book's teal/lime green scheme
TEAL = '#1a7f8e'
LIME = '#7bc42b'
GRAY = '#6b7280'
ORANGE = '#f97316'

# --- Plot 1: Acquisition curve with exponential fit ---
ax1 = axes[0, 0]
ax1.scatter(df_phase1['trial'], df_phase1['composite'],
            color=TEAL, alpha=0.7, s=40, zorder=3, label='Observed')
t_range = np.linspace(1, N_ACQUISITION, 200)
ax1.plot(t_range, neg_exp(t_range, *popt), color=ORANGE, linewidth=2,
         linestyle='--', label=f'Exp. fit (λ={lam:.3f}, R²={r_squared:.2f})')
ax1.set_xlabel('Trial')
ax1.set_ylabel('Response Strength (0-3)')
ax1.set_title('Phase 1: Acquisition Curve')
ax1.legend()
ax1.set_ylim(0, 3.2)
ax1.grid(True, alpha=0.3)

# --- Plot 2: Full timeline including recovery ---
ax2 = axes[0, 1]
p1_trials = df_phase1['trial'].values
p1_scores = df_phase1['composite'].values
ax2.plot(p1_trials, p1_scores, 'o-', color=TEAL, alpha=0.8, linewidth=1.5, label='Phase 1')

if len(df_recovery) > 0:
    rec_trials = df_recovery['trial'].values
    rec_scores = df_recovery['composite'].values
    ax2.plot(rec_trials, rec_scores, 's-', color=LIME, alpha=0.8, linewidth=1.5, label='Phase 2')
    # Mark Trial 41
    ax2.axvline(x=40.5, color=GRAY, linestyle=':', alpha=0.6)
    ax2.text(40.7, 2.8, 'REST', color=GRAY, fontsize=9)

# Mark Trial 52 (post-dishabituator)
ax2.scatter([52], [trial52_result['composite']], color=ORANGE, s=100,
            zorder=5, marker='*', label='Trial 52 (post-dishab)')

ax2.set_xlabel('Trial')
ax2.set_ylabel('Response Strength (0-3)')
ax2.set_title('Full Timeline: All Three Phases')
ax2.legend()
ax2.set_ylim(0, 3.2)
ax2.grid(True, alpha=0.3)

# --- Plot 3: Recovery comparison bar chart ---
ax3 = axes[1, 0]
labels = ['Trial 1\n(Baseline)', 'Trials 39-40\n(Late Phase 1)', 'Trial 41\n(Recovery)', 'Trial 52\n(Post-Dishab)']
values = [
    df_phase1.iloc[0]['composite'],
    np.mean(late_phase1),
    trial41_score,
    trial52_result['composite']
]
colors_bar = [TEAL, GRAY, LIME, ORANGE]
bars = ax3.bar(labels, values, color=colors_bar, alpha=0.8, edgecolor='white')
ax3.set_ylabel('Response Strength (0-3)')
ax3.set_title('Key Comparison Points')
ax3.set_ylim(0, 3.2)
for bar, val in zip(bars, values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f}', ha='center', va='bottom', fontsize=9)
ax3.grid(True, alpha=0.3, axis='y')

# --- Plot 4: Control conditions comparison ---
ax4 = axes[1, 1]
ctrl_labels = ['Phase 1\nMean', 'Baseline\n(Ctrl 1)', 'No-Agent\n(Ctrl 3)']
ctrl_values = [
    df_phase1['composite'].mean(),
    df_ctrl1['composite'].mean() if len(ctrl1_results) > 0 else 0,
    df_ctrl3['composite'].mean() if len(ctrl3_results) > 0 else 0
]
ctrl_stds = [
    df_phase1['composite'].std(),
    df_ctrl1['composite'].std() if len(ctrl1_results) > 0 else 0,
    df_ctrl3['composite'].std() if len(ctrl3_results) > 0 else 0
]
ax4.bar(ctrl_labels, ctrl_values, yerr=ctrl_stds, color=[TEAL, LIME, GRAY],
        alpha=0.8, edgecolor='white', capsize=5)
ax4.set_ylabel('Mean Composite Score (0-3)')
ax4.set_title('Control Conditions Comparison')
ax4.set_ylim(0, 3.5)
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('habituation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as habituation_results.png')

In [ ]:
# ============================================================
# SAVE RAW DATA
# ============================================================

all_data = []

# Phase 1
for r in phase1_results:
    r_clean = {k: v for k, v in r.items() if k != 'raw_response'}
    all_data.append(r_clean)

# Phase 2 / recovery
for r in recovery_results:
    r_clean = {k: v for k, v in r.items() if k != 'raw_response'}
    all_data.append(r_clean)

# Dishabituation trials
for r in [dishab_result, trial52_result]:
    r_clean = {k: v for k, v in r.items() if k != 'raw_response'}
    all_data.append(r_clean)

df_all = pd.DataFrame(all_data)
df_all.to_csv('habituation_data.csv', index=False)
print('Raw data saved to habituation_data.csv')
print(f'Total rows: {len(df_all)}')
print(df_all[['trial', 'phase', 'stimulus_type', 'composite']].tail(15))

In [ ]:
# ============================================================
# INTERPRETATION GUIDE
# ============================================================

print('='*60)
print('HOW TO INTERPRET YOUR RESULTS')
print('='*60)
print()
print('ACQUISITION CURVE (Plot 1):')
print('  lambda > 0.01 AND R² > 0.5 → Systematic decline = habituation candidate')
print('  lambda near 0 OR R² < 0.5 → No systematic decline = null result')
print()
print('SPONTANEOUS RECOVERY (Trial 41 vs. Trials 39-40):')
print('  p < 0.05 AND Trial 41 > Trials 39-40 → Recovery detected = supports habituation')
print('  p > 0.05 or Trial 41 ≤ Trials 39-40 → No recovery = consistent with fatigue')
print()
print('DISHABITUATION (Trial 52 vs. Trial 1):')
print('  Trial 52 within 1 SD of Trial 1 → Dishabituation supported')
print('  Trial 52 still near the habituated floor → Dishabituation NOT supported')
print()
print('VERDICT MATRIX:')
print('  Decline + Recovery + Dishabituation → HABITUATION criteria met')
print('  Decline + No Recovery + No Dishabituation → FATIGUE or FILTERING')
print('  No decline → NULL RESULT (stimulus may be too salient, or model genuinely robust)')
print()
print('Remember: compare your result to your PRE-REGISTERED prediction above.')
print('A result that matches your prediction is far more informative than one you found after looking.')

## Deliverable

Write a short report (300–500 words) that includes:

1. **What you predicted** (copy from your pre-registration cell above)
2. **What you found** (λ value, R², spontaneous recovery p-value and Cohen's d, dishabituation result)
3. **Verdict**: Does the agent meet the habituation criteria? Which criteria passed? Which failed?
4. **Interpretation**: If positive — what is the operational implication for a real deployment? If negative — which alternative (fatigue, filtering, null) is most consistent, and what would you do differently?
5. **One extension**: What is the single most important follow-up experiment, and why?

---

## Extension Ideas

- **Frequency effect test**: Run three parallel acquisition curves at different inter-trial intervals (30s, 5min, 1hr). Does faster repetition produce faster habituation?
- **Intensity effect test**: Compare acquisition curves for a low-salience alert vs. a medium-salience alert. Does the weaker one habituate faster?
- **Potentiation test**: Run Phase 1 twice on consecutive days. Is the second session's habituation curve faster?
- **Multiple models**: Run the protocol on GPT-4o, Claude 3.5, and Gemini 1.5. Do they habituate at different rates? Do any fail to habituate?
- **Sensitization**: Instead of repeated identical stimuli, try repeated escalating stimuli. Does the response grow instead of shrink? That is sensitization — the other direction on the learning ladder.